<h1>Silver GTFS<h1>

<h2>Imports<h2>

<h3>Spark and Initialization<h3>

In [32]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Warsaw_Bus_Project") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate()

<h3>Imports<h3>

In [33]:
from datetime import datetime , timedelta
import sys
import gc
sys.path.append('../work')
from config import db_properties,DB_CONFIG,jdbc_url
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
import pandas as pd
import numpy as np
import pyspark.pandas as ps
from pyspark.sql.types import FloatType, IntegerType, BooleanType
from pyspark.sql import Row
api_key = os.getenv("WARSAW_API_KEY")
import psycopg2
from psycopg2 import sql
import pyspark.sql.functions as sf
import zipfile
import urllib.request
from pyspark.testing import assertDataFrameEqual
from pyspark.sql import functions as F
import geopandas as gpd
from shapely.geometry import Point

<h3>Creating Silver schema<h3>

In [34]:
conn = psycopg2.connect(dbname="Warsaw_Bus_DB", user="admin", password="admin", host="postgres")
cur = conn.cursor()
cur.execute("CREATE SCHEMA IF NOT EXISTS silver;")
conn.commit()
conn.close()

<h2>Silver Stops Table<h2>

<h3>Extracting Bronze Stops<h3>

In [35]:
df_stops_bronze = spark.read.jdbc(url=jdbc_url, table="bronze.stops", properties=db_properties)

<h3>Transforming Stops data<h3>

In [36]:
df_stops_silver = df_stops_bronze\
                .withColumn("stop_id",F.col("stop_id").cast(IntegerType()))\
                .withColumn("stop_lat",F.col("stop_lat").cast(FloatType()))\
                .withColumn("stop_lon",F.col("stop_lon").cast(FloatType()))


<h3>Adding District to Data<h3>

In [37]:
pdf_stops = df_stops_silver.select("stop_id", "stop_lat", "stop_lon").toPandas()
geometry = [Point(xy) for xy in zip(pdf_stops.stop_lon, pdf_stops.stop_lat)]
gdf_stops = gpd.GeoDataFrame(pdf_stops, geometry=geometry, crs="EPSG:4326")
gdf_districts = gpd.read_file("../Silver/warszawa-dzielnice.geojson")
gdf_stops_with_districts = gpd.sjoin(gdf_stops, gdf_districts, how="left", predicate="within")
df_districts = spark.createDataFrame(gdf_stops_with_districts[["stop_id", "name"]].rename(columns={"name": "district"}))
df_stops_silver = df_stops_silver.join(df_districts, on="stop_id", how="left")
df_stops_silver = df_stops_silver.where(F.col("district")!="Warszawa")

<h3>Writing to DB<h3>

In [38]:
df_stops_silver.write.jdbc(url=jdbc_url, table="silver.stops", mode="overwrite", properties=db_properties)

<h2>Silver Routes Table<h2>

<h3>Extracting Bronze Routes<h3>

In [39]:
df_routes_bronze = spark.read.jdbc(url=jdbc_url, table="bronze.routes", properties=db_properties)
df_routes_bronze.show(5)

+--------+---------+----------------+---------------+--------------------+----------+
|route_id|agency_id|route_short_name|route_long_name|          route_desc|route_type|
+--------+---------+----------------+---------------+--------------------+----------+
|    0_A1|        5|              A1|           NULL|WARSZAWA ŚRÓDMIEŚ...|         2|
|   0_E-1|        2|             E-1|           NULL|METRO STADION NAR...|         3|
|   0_E-2|        2|             E-2|           NULL|METRO RATUSZ ARSE...|         3|
|   0_L-1|        2|             L-1|           NULL|PIASECZNO PKP PIA...|         3|
|   0_L10|        2|             L10|           NULL|LEGIONOWO PKP LEG...|         3|
+--------+---------+----------------+---------------+--------------------+----------+
only showing top 5 rows



<h3>Transforming Routes data<h3>

In [40]:
df_routes_bronze = df_routes_bronze.drop("route_long_name")

In [41]:
df_routes_silver = df_routes_bronze\
                .withColumn("agency_id",F.col("agency_id").cast(IntegerType()))\
                .withColumn("route_type",F.col("route_type").cast(IntegerType()))\
                .withColumnRenamed("route_short_name", "route_name")
df_routes_silver

DataFrame[route_id: string, agency_id: int, route_name: string, route_desc: string, route_type: int]

<h3>Writing to DB<h3>

In [42]:
df_routes_silver.write.jdbc(url=jdbc_url, table="silver.routes", mode="overwrite", properties=db_properties)

<h2>Silver Trips Table<h2>

<h3>Extracting Bronze Trips<h3>

In [43]:
df_trips_bronze = spark.read.jdbc(url=jdbc_url, table="bronze.trips", properties=db_properties)
df_trips_bronze.show(5)

+--------+----------+-----------+--------------------+------------+--------+
|route_id|service_id|    trip_id|       trip_headsign|direction_id|shape_id|
+--------+----------+-----------+--------------------+------------+--------+
|    5_24|       5_2|5_2_5620563|24 -> Rondo Daszy...|           0|  317419|
|    5_24|       5_2|5_2_5620567|24 -> Rondo Daszy...|           0|  317419|
|    5_24|       5_2|5_2_5620573|24 -> Rondo Daszy...|           0|  317419|
|    5_24|       5_2|5_2_5620577|24 -> Rondo Daszy...|           0|  317419|
|    5_24|       5_2|5_2_5620581|24 -> Rondo Daszy...|           0|  317419|
+--------+----------+-----------+--------------------+------------+--------+
only showing top 5 rows



<h3>Transforming Stops data<h3>

In [44]:
df_trips_bronze = df_trips_bronze.drop("shape_id")

In [45]:
df_trips_silver = df_trips_bronze\
                .withColumn("direction_id",F.col("direction_id").cast(IntegerType()))
df_trips_silver

DataFrame[route_id: string, service_id: string, trip_id: string, trip_headsign: string, direction_id: int]

<h3>Writing to DB<h3>

In [46]:
df_trips_silver.write.jdbc(url=jdbc_url, table="silver.trips", mode="overwrite", properties=db_properties)

<h2>Silver Calendar Table<h2>

<h3>Extracting Bronze Calendar<h3>

In [47]:
df_calendar_bronze = spark.read.jdbc(url=jdbc_url, table="bronze.calendar", properties=db_properties)
df_calendar_bronze.show()

+----------+------+-------+---------+--------+------+--------+------+----------+--------+
|service_id|monday|tuesday|wednesday|thursday|friday|saturday|sunday|start_date|end_date|
+----------+------+-------+---------+--------+------+--------+------+----------+--------+
|       0_2|     0|      0|        0|       1|     0|       0|     0|  20260409|20260409|
|       1_5|     0|      0|        0|       0|     1|       0|     0|  20260410|20260410|
|       2_3|     0|      0|        0|       0|     0|       1|     0|  20260411|20260411|
|       3_1|     0|      0|        0|       0|     0|       0|     1|  20260412|20260412|
|       4_2|     1|      0|        0|       0|     0|       0|     0|  20260413|20260413|
|       5_2|     0|      1|        0|       0|     0|       0|     0|  20260414|20260414|
|       6_2|     0|      0|        1|       0|     0|       0|     0|  20260415|20260415|
|       7_2|     0|      0|        0|       1|     0|       0|     0|  20260416|20260416|
|       8_

<h3>Transforming Stops data<h3>

In [48]:
days = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
for day in days:
    df_calendar_bronze = df_calendar_bronze.withColumn(day, F.col(day).cast(BooleanType()))
df_calendar_silver = df_calendar_bronze

In [49]:
df_calendar_silver 

DataFrame[service_id: string, monday: boolean, tuesday: boolean, wednesday: boolean, thursday: boolean, friday: boolean, saturday: boolean, sunday: boolean, start_date: string, end_date: string]

<h3>Writing to DB<h3>

In [50]:
df_calendar_silver.write.jdbc(url=jdbc_url, table="silver.calendar", mode="overwrite", properties=db_properties)

<h2>Silver Stop_Times Table<h2>

<h3>Extracting Bronze Stop_Times<h3>

In [51]:
df_stop_times_bronze = spark.read.jdbc(url=jdbc_url, table="bronze.stop_times", properties=db_properties)
df_stop_times_bronze

DataFrame[trip_id: string, arrival_time: string, departure_time: string, stop_id: string, stop_sequence: string, stop_headsign: string, pickup_type: string, drop_off_type: string, shape_dist_traveled: string, timepoint: string]

<h3>Transforming Stop_Times data</h3>

In [52]:
df_stop_times_bronze = df_stop_times_bronze.drop("stop_headsign")
df_stop_times_bronze = df_stop_times_bronze\
                .withColumn("stop_sequence",F.col("stop_sequence").cast(IntegerType()))\
                .withColumn("pickup_type",F.col("pickup_type").cast(IntegerType()))\
                .withColumn("drop_off_type",F.col("drop_off_type").cast(IntegerType()))\
                .withColumn("stop_id",F.col("stop_id").cast(IntegerType()))\
                .withColumn("timepoint",F.col("timepoint").cast(IntegerType()))\
                .withColumn("shape_dist_traveled",F.col("shape_dist_traveled").cast(FloatType()))
df_stop_times_bronze

DataFrame[trip_id: string, arrival_time: string, departure_time: string, stop_id: int, stop_sequence: int, pickup_type: int, drop_off_type: int, shape_dist_traveled: float, timepoint: int]

In [53]:
df_stop_times_bronze.show(5)

+-----------+------------+--------------+-------+-------------+-----------+-------------+-------------------+---------+
|    trip_id|arrival_time|departure_time|stop_id|stop_sequence|pickup_type|drop_off_type|shape_dist_traveled|timepoint|
+-----------+------------+--------------+-------+-------------+-----------+-------------+-------------------+---------+
|6_2_5563505|    17:12:00|      17:12:00|   2245|           26|          3|            3|             11.474|        1|
|6_2_5563505|    17:13:00|      17:13:00|   3080|           27|          3|            3|             11.784|        1|
|6_2_5563505|    17:15:00|      17:15:00|   2002|           28|          3|            3|             12.324|        1|
|6_2_5563505|    17:18:00|      17:18:00|   2006|           29|          0|            0|             12.916|        1|
|6_2_5563505|    17:19:00|      17:19:00|   2007|           30|          0|            0|             13.353|        1|
+-----------+------------+--------------

<h3>Setting up timestamps<h3>

In [57]:
df_stop_times_joined = df_stop_times_bronze.join(df_trips_silver, on="trip_id", how="inner")\
                .join(df_calendar_silver, on="service_id", how="inner")

In [58]:
df_stop_times_joined.show(5)

+----------+-----------+------------+--------------+-------+-------------+-----------+-------------+-------------------+---------+--------+-----------------+------------+------+-------+---------+--------+------+--------+------+----------+--------+
|service_id|    trip_id|arrival_time|departure_time|stop_id|stop_sequence|pickup_type|drop_off_type|shape_dist_traveled|timepoint|route_id|    trip_headsign|direction_id|monday|tuesday|wednesday|thursday|friday|saturday|sunday|start_date|end_date|
+----------+-----------+------------+--------------+-------+-------------+-----------+-------------+-------------------+---------+--------+-----------------+------------+------+-------+---------+--------+------+--------+------+----------+--------+
|       7_2|7_2_2261191|    06:35:00|      06:35:00|   2839|            0|          0|            1|                0.0|        1|   7_139|139 -> Os. Kabaty|           0| false|  false|    false|    true| false|   false| false|  20260416|20260416|
|       

In [63]:
df_stop_times_silver = df_stop_times_joined \
    .withColumn("op_date", F.to_date(F.col("start_date"), "yyyyMMdd")) \
    .withColumn("arrival_sec", F.col("arrival_time").substr(1, 2).cast(IntegerType()) * 3600 + \
                                F.col("arrival_time").substr(4, 2).cast(IntegerType()) * 60 + \
                                F.col("arrival_time").substr(7, 2).cast(IntegerType())) \
    .withColumn("departure_sec", F.col("departure_time").substr(1, 2).cast(IntegerType()) * 3600 + \
                                F.col("departure_time").substr(4, 2).cast(IntegerType()) * 60 + \
                                F.col("departure_time").substr(7, 2).cast(IntegerType())) \
    .withColumn("parsed_arrival_date",
                F.expr("cast(to_timestamp(op_date) + make_interval(0, 0, 0, 0, 0, 0, arrival_sec) as timestamp)")) \
    .withColumn("parsed_departure_date",
                F.expr("cast(to_timestamp(op_date) + make_interval(0, 0, 0, 0, 0, 0, departure_sec) as timestamp)"))

In [ ]:
df_stop_times_silver = df_stop_times_silver.select(
    "trip_id", "stop_id", "stop_sequence", "pickup_type", 
    "drop_off_type", "shape_dist_traveled", 
     F.col("parsed_arrival_date").alias("arrival_time"), F.col("parsed_departure_date").alias("departure_time"))
df_stop_times_silver

DataFrame[trip_id: string, stop_id: int, stop_sequence: int, pickup_type: int, drop_off_type: int, shape_dist_traveled: float, arrival_datetime: timestamp, departure_datetime: timestamp]

In [66]:
df_stop_times_silver.select("arrival_datetime", "departure_datetime").show(5)

+-------------------+-------------------+
|   arrival_datetime| departure_datetime|
+-------------------+-------------------+
|2026-04-16 06:35:00|2026-04-16 06:35:00|
|2026-04-16 06:36:00|2026-04-16 06:36:00|
|2026-04-16 06:37:00|2026-04-16 06:37:00|
|2026-04-16 06:39:00|2026-04-16 06:39:00|
|2026-04-16 06:40:00|2026-04-16 06:40:00|
+-------------------+-------------------+
only showing top 5 rows



In [ ]:
print(df_stop_times_silver.select("arrival_datetime", "departure_datetime").dtypes)
df_stop_times_silver.select("arrival_datetime", "departure_datetime").limit(3).show(truncate=False)